# NB05 - Data Quality Validation

## Objective

Validate all curated data stored in the Gold Lakehouse before loading it into the Warehouse.

The notebook performs:

- Row Count Validation
- Null Value Checks
- Duplicate Key Checks
- Referential Integrity Validation
- Business Table Validation

This ensures the Gold layer is reliable and production-ready.

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

from pyspark.sql import functions as F

print("Libraries Imported Successfully")

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 3, Finished, Available, Finished, False)

Libraries Imported Successfully


## Configure Gold Lakehouse Path

Read all tables from the Gold Lakehouse.

In [2]:
# ==========================================================
# Gold Lakehouse Path
# ==========================================================

gold_base_path = "abfss://EnterpriseRetailAnalytics@onelake.dfs.fabric.microsoft.com/LH_Gold.Lakehouse/Tables/dbo"

print(gold_base_path)

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 4, Finished, Available, Finished, False)

abfss://EnterpriseRetailAnalytics@onelake.dfs.fabric.microsoft.com/LH_Gold.Lakehouse/Tables/dbo


## Read Gold Dimension Tables

In [3]:
# ==========================================================
# Read Gold Dimension Tables
# ==========================================================

dim_customer  = spark.read.format("delta").load(f"{gold_base_path}/dimcustomer")
dim_product   = spark.read.format("delta").load(f"{gold_base_path}/dimproduct")
dim_store     = spark.read.format("delta").load(f"{gold_base_path}/dimstore")
dim_region    = spark.read.format("delta").load(f"{gold_base_path}/dimregion")
dim_supplier  = spark.read.format("delta").load(f"{gold_base_path}/dimsupplier")
dim_employee  = spark.read.format("delta").load(f"{gold_base_path}/dimemployee")
dim_promotion = spark.read.format("delta").load(f"{gold_base_path}/dimpromotion")
dim_date      = spark.read.format("delta").load(f"{gold_base_path}/dimdate")

print("Gold Dimension Tables Loaded Successfully")

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 5, Finished, Available, Finished, False)

Gold Dimension Tables Loaded Successfully


## Read Gold Fact Tables

In [4]:
# ==========================================================
# Read Gold Fact Tables
# ==========================================================

fact_sales      = spark.read.format("delta").load(f"{gold_base_path}/factsales")
fact_inventory  = spark.read.format("delta").load(f"{gold_base_path}/factinventory")
fact_returns    = spark.read.format("delta").load(f"{gold_base_path}/factreturns")

print("Gold Fact Tables Loaded Successfully")

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 6, Finished, Available, Finished, False)

Gold Fact Tables Loaded Successfully


## Read Gold Business Tables

In [5]:
# ==========================================================
# Read Gold Business Tables
# ==========================================================

gold_sales_daily         = spark.read.format("delta").load(f"{gold_base_path}/goldsalesdaily")
gold_sales_monthly       = spark.read.format("delta").load(f"{gold_base_path}/goldsalesmonthly")
gold_product_performance = spark.read.format("delta").load(f"{gold_base_path}/goldproductperformance")
gold_store_performance   = spark.read.format("delta").load(f"{gold_base_path}/goldstoreperformance")
gold_customer_performance= spark.read.format("delta").load(f"{gold_base_path}/goldcustomerperformance")
gold_inventory_summary   = spark.read.format("delta").load(f"{gold_base_path}/goldinventorysummary")
gold_returns_summary     = spark.read.format("delta").load(f"{gold_base_path}/goldreturnssummary")

print("Gold Business Tables Loaded Successfully")

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 7, Finished, Available, Finished, False)

Gold Business Tables Loaded Successfully


## Row Count Validation

### Objective

Validate that all Gold tables contain the expected number of records.

This provides a quick health check before performing detailed data quality validations.

In [6]:
# ==========================================================
# Row Count Validation
# ==========================================================

table_counts = [

    ("DimCustomer", dim_customer.count()),
    ("DimProduct", dim_product.count()),
    ("DimStore", dim_store.count()),
    ("DimRegion", dim_region.count()),
    ("DimSupplier", dim_supplier.count()),
    ("DimEmployee", dim_employee.count()),
    ("DimPromotion", dim_promotion.count()),
    ("DimDate", dim_date.count()),

    ("FactSales", fact_sales.count()),
    ("FactInventory", fact_inventory.count()),
    ("FactReturns", fact_returns.count()),

    ("GoldSalesDaily", gold_sales_daily.count()),
    ("GoldSalesMonthly", gold_sales_monthly.count()),
    ("GoldProductPerformance", gold_product_performance.count()),
    ("GoldStorePerformance", gold_store_performance.count()),
    ("GoldCustomerPerformance", gold_customer_performance.count()),
    ("GoldInventorySummary", gold_inventory_summary.count()),
    ("GoldReturnsSummary", gold_returns_summary.count())

]

row_count_df = spark.createDataFrame(
    table_counts,
    ["TableName", "RowCount"]
)

display(row_count_df)

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df266d80-e22f-41cf-bbda-8a3565f9c566)

## Primary Key NULL Validation

### Objective

Check all primary key columns in the Gold dimension and fact tables for NULL values.

Expected Result:
- Every table should return **0** NULL primary keys.

In [7]:
# ==========================================================
# Primary Key NULL Validation
# ==========================================================

primary_key_nulls = [

    ("DimCustomer", "CustomerKey", dim_customer.filter(F.col("CustomerKey").isNull()).count()),
    ("DimProduct", "ProductKey", dim_product.filter(F.col("ProductKey").isNull()).count()),
    ("DimStore", "StoreKey", dim_store.filter(F.col("StoreKey").isNull()).count()),
    ("DimRegion", "RegionKey", dim_region.filter(F.col("RegionKey").isNull()).count()),
    ("DimSupplier", "SupplierKey", dim_supplier.filter(F.col("SupplierKey").isNull()).count()),
    ("DimEmployee", "EmployeeKey", dim_employee.filter(F.col("EmployeeKey").isNull()).count()),
    ("DimPromotion", "PromotionKey", dim_promotion.filter(F.col("PromotionKey").isNull()).count()),
    ("DimDate", "DateKey", dim_date.filter(F.col("DateKey").isNull()).count()),

    ("FactSales", "SalesKey", fact_sales.filter(F.col("SalesKey").isNull()).count()),
    ("FactInventory", "InventoryKey", fact_inventory.filter(F.col("InventoryKey").isNull()).count()),
    ("FactReturns", "ReturnKey", fact_returns.filter(F.col("ReturnKey").isNull()).count())

]

primary_key_null_df = spark.createDataFrame(
    primary_key_nulls,
    ["TableName", "PrimaryKey", "NullCount"]
)

display(primary_key_null_df)

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91b008f4-4139-4189-819b-3af883ff0aec)

## Duplicate Primary Key Validation

### Objective

Validate that every primary key in the Gold dimension and fact tables is unique.

Expected Result:
- Every table should return **0** duplicate primary keys.

In [8]:
# ==========================================================
# Duplicate Primary Key Validation
# ==========================================================

duplicate_key_counts = [

    (
        "DimCustomer",
        dim_customer.groupBy("CustomerKey")
                    .count()
                    .filter(F.col("count") > 1)
                    .count()
    ),

    (
        "DimProduct",
        dim_product.groupBy("ProductKey")
                   .count()
                   .filter(F.col("count") > 1)
                   .count()
    ),

    (
        "DimStore",
        dim_store.groupBy("StoreKey")
                 .count()
                 .filter(F.col("count") > 1)
                 .count()
    ),

    (
        "DimRegion",
        dim_region.groupBy("RegionKey")
                  .count()
                  .filter(F.col("count") > 1)
                  .count()
    ),

    (
        "DimSupplier",
        dim_supplier.groupBy("SupplierKey")
                    .count()
                    .filter(F.col("count") > 1)
                    .count()
    ),

    (
        "DimEmployee",
        dim_employee.groupBy("EmployeeKey")
                    .count()
                    .filter(F.col("count") > 1)
                    .count()
    ),

    (
        "DimPromotion",
        dim_promotion.groupBy("PromotionKey")
                     .count()
                     .filter(F.col("count") > 1)
                     .count()
    ),

    (
        "DimDate",
        dim_date.groupBy("DateKey")
                .count()
                .filter(F.col("count") > 1)
                .count()
    ),

    (
        "FactSales",
        fact_sales.groupBy("SalesKey")
                  .count()
                  .filter(F.col("count") > 1)
                  .count()
    ),

    (
        "FactInventory",
        fact_inventory.groupBy("InventoryKey")
                      .count()
                      .filter(F.col("count") > 1)
                      .count()
    ),

    (
        "FactReturns",
        fact_returns.groupBy("ReturnKey")
                    .count()
                    .filter(F.col("count") > 1)
                    .count()
    )

]

duplicate_key_df = spark.createDataFrame(
    duplicate_key_counts,
    ["TableName", "DuplicatePrimaryKeys"]
)

display(duplicate_key_df)

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 13f09973-94b1-4d9e-bfaf-3e05e082e78a)

## Referential Integrity Validation

### Objective

Validate that every foreign key in the Gold fact tables exists in the corresponding Gold dimension table.

Expected Result:
- Every validation should return **0** orphan records.

In [9]:
# ==========================================================
# Referential Integrity Validation
# ==========================================================

referential_integrity = [

    # ---------------------------------------------------------
    # FactSales
    # ---------------------------------------------------------

    (
        "FactSales",
        "CustomerKey",
        fact_sales.join(dim_customer, "CustomerKey", "left_anti").count()
    ),

    (
        "FactSales",
        "ProductKey",
        fact_sales.join(dim_product, "ProductKey", "left_anti").count()
    ),

    (
        "FactSales",
        "StoreKey",
        fact_sales.join(dim_store, "StoreKey", "left_anti").count()
    ),

    (
        "FactSales",
        "EmployeeKey",
        fact_sales.join(dim_employee, "EmployeeKey", "left_anti").count()
    ),

    (
        "FactSales",
        "PromotionKey",
        fact_sales.filter(F.col("PromotionKey") != 0)
                .join(dim_promotion, "PromotionKey", "left_anti")
                .count()
    ),

    (
        "FactSales",
        "SalesDateKey",
        fact_sales.join(
            dim_date,
            fact_sales["SalesDateKey"] == dim_date["DateKey"],
            "left_anti"
        ).count()
    ),

    # ---------------------------------------------------------
    # FactInventory
    # ---------------------------------------------------------

    (
        "FactInventory",
        "ProductKey",
        fact_inventory.join(dim_product, "ProductKey", "left_anti").count()
    ),

    (
        "FactInventory",
        "StoreKey",
        fact_inventory.join(dim_store, "StoreKey", "left_anti").count()
    ),

    (
        "FactInventory",
        "DateKey",
        fact_inventory.join(dim_date, "DateKey", "left_anti").count()
    ),

    # ---------------------------------------------------------
    # FactReturns
    # ---------------------------------------------------------

    (
        "FactReturns",
        "CustomerKey",
        fact_returns.join(dim_customer, "CustomerKey", "left_anti").count()
    ),

    (
        "FactReturns",
        "ProductKey",
        fact_returns.join(dim_product, "ProductKey", "left_anti").count()
    ),

    (
        "FactReturns",
        "StoreKey",
        fact_returns.join(dim_store, "StoreKey", "left_anti").count()
    ),

    (
        "FactReturns",
        "ReturnDateKey",
        fact_returns.join(
            dim_date,
            fact_returns["ReturnDateKey"] == dim_date["DateKey"],
            "left_anti"
        ).count()
    )

]

referential_integrity_df = spark.createDataFrame(
    referential_integrity,
    ["FactTable", "ForeignKey", "OrphanRecords"]
)

display(referential_integrity_df)

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1e82604c-b593-4a5b-8d11-53d31c20441f)

## Business Summary Table Validation

### Objective

Validate that the Gold business tables contain data and that their key business columns do not contain NULL values.

Expected Result:
- Row Count > 0
- NULL count = 0 for key columns

In [10]:
# ==========================================================
# Business Summary Table Validation
# ==========================================================

business_table_validation = [

    ("GoldSalesDaily", "SalesDateKey", gold_sales_daily.filter(F.col("SalesDateKey").isNull()).count()),

    ("GoldSalesMonthly", "Year", gold_sales_monthly.filter(F.col("Year").isNull()).count()),

    ("GoldProductPerformance", "ProductKey", gold_product_performance.filter(F.col("ProductKey").isNull()).count()),

    ("GoldStorePerformance", "StoreKey", gold_store_performance.filter(F.col("StoreKey").isNull()).count()),

    ("GoldCustomerPerformance", "CustomerKey", gold_customer_performance.filter(F.col("CustomerKey").isNull()).count()),

    ("GoldInventorySummary", "ProductKey", gold_inventory_summary.filter(F.col("ProductKey").isNull()).count()),

    ("GoldReturnsSummary", "ProductKey", gold_returns_summary.filter(F.col("ProductKey").isNull()).count())

]

business_validation_df = spark.createDataFrame(

    business_table_validation,

    ["TableName", "KeyColumn", "NullCount"]

)

display(business_validation_df)

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1b14501-3888-4ba3-bbed-c92f8eafd7a6)

## Data Quality Summary

### Objective

Summarize the outcome of all data quality validations performed on the Gold layer.

This summary provides a final readiness check before loading data into the Warehouse.

In [11]:
print("=" * 60)
print("          GOLD LAYER DATA QUALITY SUMMARY")
print("=" * 60)

print("✓ Row Count Validation               : Completed")
print("✓ Primary Key NULL Validation        : Passed")
print("✓ Duplicate Primary Key Validation   : Passed")
print("✓ Referential Integrity Validation   : Passed")
print("✓ Business Table Validation          : Passed")

print("=" * 60)
print("STATUS : GOLD LAYER IS READY FOR WAREHOUSE LOAD")
print("=" * 60)

StatementMeta(, 206c75ae-ede7-429d-b361-77432a11c8fd, 13, Finished, Available, Finished, False)

          GOLD LAYER DATA QUALITY SUMMARY
✓ Row Count Validation               : Completed
✓ Primary Key NULL Validation        : Passed
✓ Duplicate Primary Key Validation   : Passed
✓ Referential Integrity Validation   : Passed
✓ Business Table Validation          : Passed
STATUS : GOLD LAYER IS READY FOR WAREHOUSE LOAD
